## 1. Imports and Dependencies

In [1]:
import math
import random
import re

import torch
import torch.nn as nn
from torch.nn import functional as F

from transformers import GPT2TokenizerFast

import os
import gc

/home/usera19/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
CUDA version: 13.0


## 2. Text Preprocessing

Transformation pipeline to remove artifacts, enforce boundaries and whitespace to produce input space for language modeling.

In [4]:
class TextCleaner:
    def __init__(self, filepath):
        self.filepath = filepath
        self.text = None

    def load(self):
        with open(self.filepath, "r", encoding="utf-8") as f:
            self.text = f.read()
        print(f"[INFO] Loaded dataset : {len(self.text):,} chars")
        return self

    def fix_artifacts(self):
        self.text = self.text.replace("@-@", "-")
        self.text = self.text.replace("@,@", ",")
        self.text = self.text.replace("@.@", ".")
        self.text = self.text.replace("<unk>", "")
        print("[INFO] Artifacts fixed")
        return self

    def mark_section_boundaries(self):
        self.text = re.sub(r"=+.*?=+", " <|endoftext|> ", self.text)
        print("[INFO] Section boundaries marked")
        return self

    def clean_lines(self):
        lines = self.text.split("\n")
        cleaned = [line.strip() for line in lines if line.strip()]
        self.text = "\n".join(cleaned)
        print("[INFO] Lines cleaned")
        return self

    def normalize_whitespace(self):
        self.text = re.sub(r"[ \t]+", " ", self.text)
        self.text = re.sub(r"\n{3,}", "\n\n", self.text)
        self.text = self.text.replace("\n\n", " <|endoftext|> ")
        self.text = self.text.strip()
        print("[INFO] Whitespace normalized")
        return self

    def run(self):
        self.load()
        self.fix_artifacts()
        self.mark_section_boundaries()
        self.clean_lines()
        self.normalize_whitespace()
        return self

    def report(self, n=300):
        print(f"\n[INFO] Final text size : {len(self.text):,} chars")
        print(f"[INFO] Sample          :\n{self.text[:n]}")
        return self

    def get_text(self):
        if self.text is None:
            raise RuntimeError("TextCleaner: call .run() before .get_text()")
        return self.text


## 3. Tokenization as a Discrete Encoding Operator over Text Space

Implements a function mapping text inputs into sequences over a finite vocabulary. Ensures consistency via sanity checks and evaluates structural properties of the resulting token distribution.

In [5]:
class TokenizerWrapper:
    def __init__(self, model_name="gpt2"):
        self.model_name  = model_name
        self.tokenizer   = None
        self.vocab_size  = None

    def load(self):
        self.tokenizer = GPT2TokenizerFast.from_pretrained(self.model_name)
        self.tokenizer.model_max_length = 10**9

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        self.vocab_size = self.tokenizer.vocab_size

        print(f"[INFO] Tokenizer loaded  : {self.model_name}")
        print(f"[INFO] Vocab size        : {self.vocab_size:,}")
        print(f"[INFO] EOS token ID      : {self.tokenizer.eos_token_id}")
        return self

    def encode(self, s):
        if self.tokenizer is None:
            raise RuntimeError("TokenizerWrapper: call .load() first")
        return self.tokenizer.encode(s, add_special_tokens=False)

    def decode(self, tokens):
        if self.tokenizer is None:
            raise RuntimeError("TokenizerWrapper: call .load() first")
        return self.tokenizer.decode(tokens, skip_special_tokens=True)

    def encode_chunked(self, text, chunk_size=100000):
        if self.tokenizer is None:
            raise RuntimeError("TokenizerWrapper: call .load() first")

        tokens = []
        total  = len(text)
        for i in range(0, total, chunk_size):
            chunk = text[i:i+chunk_size]

            if i + chunk_size < total:
                last_space = chunk.rfind(" ")
                if last_space != -1:
                    chunk = chunk[:last_space]

            tokens.extend(
                self.tokenizer.encode(chunk, add_special_tokens=False)
            )

            pct = min(100, int((i + chunk_size) / total * 100))
            print(f"\r[INFO] Encoding... {pct:3d}%", end="", flush=True)

        print()
        return tokens

    def sanity_check(self):
        test   = "hello world <|endoftext|> next document"
        tokens = self.encode(test)
        decoded = self.decode(tokens)
        print(f"[INFO] Sanity check")
        print(f"       input   : {test}")
        print(f"       tokens  : {tokens}")
        print(f"       decoded : {decoded}")
        assert self.tokenizer.eos_token_id == 50256, \
            "EOS token ID mismatch — expected 50256"
        print(f"       EOS ID  : {self.tokenizer.eos_token_id} ✓")
        return self

    def analyse(self, text, encoded):
        words         = text.split()
        num_words     = len(words)
        num_tokens    = len(encoded)
        unique_words  = len(set(words))
        unique_tokens = len(set(encoded))
        avg_tok_word  = num_tokens / max(num_words, 1)
        avg_word_len  = sum(len(w) for w in words) / max(num_words, 1)

        print(f"\n===== DATASET ANALYSIS =====")
        print(f"Total Words           : {num_words:,}")
        print(f"Total Tokens (BPE)    : {num_tokens:,}")
        print(f"Unique Words          : {unique_words:,}")
        print(f"Unique Tokens         : {unique_tokens:,}")
        print(f"Avg Tokens per Word   : {avg_tok_word:.2f}")
        print(f"Avg Word Length       : {avg_word_len:.2f}")
        return self

    def run(self, text):
        self.load()
        self.sanity_check()
        print("\n[INFO] Encoding text in chunks...")
        encoded = self.encode_chunked(text)
        self.analyse(text, encoded)
        return encoded

## 4. Dataset Partitioning
Splits token data into training and validation sets.

In [6]:
class DataManager:
    def __init__(self, data):
        self.data = data
        self.train_data = None
        self.val_data = None

    def split(self, ratio=0.9):
        n = int(ratio * len(self.data))
        self.train_data = self.data[:n].clone()
        self.val_data   = self.data[n:].clone()

        del self.data
        gc.collect()
        torch.cuda.empty_cache()

        print(f"Train tokens : {len(self.train_data):,}")
        print(f"Val tokens   : {len(self.val_data):,}")
        print(f"Train size   : {self.train_data.element_size() * len(self.train_data) / 1e9:.2f} GB")
        print(f"Val size     : {self.val_data.element_size() * len(self.val_data) / 1e9:.2f} GB")

        return self

    def get_data(self):
        return self.train_data, self.val_data

## 5. Model Configuration
Defines hyperparameters for training and model architecture.

In [7]:
class Config:
    def __init__(self):
        self.batch_size    = 16
        self.block_size    = 256
        self.max_iters     = 25000
        self.learning_rate = 2e-4
        self.n_embd        = 512
        self.n_head        = 8
        self.n_layer       = 8
        self.grad_accum    = 4
        self.dropout       = 0.1
        self.eval_interval = 500
        self.eval_iters    = 100
        self.warmup_steps  = 1000
        self.patience      = 10
        self.device        = "cuda" if torch.cuda.is_available() else "cpu"

    def summary(self):
        print("\n===== CONFIG =====")
        for k, v in vars(self).items():
            print(f"{k:15}: {v}")
        return self

## 6. Hardware Monitoring
Tracks GPU usage and manages memory during training.

In [8]:
class HardwareManager:
    def __init__(self):
        pass

    def gpu_usage(self):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved  = torch.cuda.memory_reserved() / 1e9
            total     = torch.cuda.get_device_properties(0).total_memory / 1e9
            free      = total - allocated
            print(f"GPU allocated : {allocated:.2f} GB")
            print(f"GPU reserved  : {reserved:.2f} GB")
            print(f"GPU free      : {free:.2f} GB")
        else:
            print("Running on CPU")

    def info(self):
        print("\n===== HARDWARE INFO =====")
        if torch.cuda.is_available():
            print("GPU        :", torch.cuda.get_device_name(0))
            print("CUDA ver   :", torch.version.cuda)
            print("Total VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
        self.gpu_usage()
        return self

    def reset_gpu(self):
        to_delete = [
            k for k, v in globals().items()
            if isinstance(v, (torch.nn.Module, torch.Tensor))
            and k not in ("train_data", "val_data")
        ]
        for k in to_delete:
            try:
                globals()[k] = globals()[k].cpu()
            except Exception:
                pass
            del globals()[k]
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("GPU reset complete")
        self.gpu_usage()
        return self

In [9]:
class BatchLoader:
    def __init__(self, train_data, val_data, block_size, batch_size, device):
        self.train_data = train_data
        self.val_data   = val_data
        self.block_size = block_size
        self.batch_size = batch_size
        self.device     = device

    def get_batch(self, split):
        src = self.train_data if split == "train" else self.val_data
        ix = torch.randint(0, len(src) - self.block_size - 1, (self.batch_size,))
        x = src[ix.unsqueeze(1) + torch.arange(self.block_size)]
        y = src[ix.unsqueeze(1) + torch.arange(1, self.block_size + 1)]
        return x.to(self.device), y.to(self.device)



In [10]:
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer(
            "tril",
            torch.tril(torch.ones(block_size, block_size)).bool()
        )
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(~self.tril[:T, :T], float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.attn_dropout(wei)
        v = self.value(x)
        return wei @ v

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj  = nn.Linear(n_embd, n_embd)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.resid_dropout(self.proj(out))



In [12]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [13]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa  = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff  = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [14]:
class GPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout, device):
        super().__init__()
        self.device = device
        self.block_size = block_size

        self.token_embedding_table    = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])

        self.ln_f    = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)

        self.lm_head.weight = self.token_embedding_table.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_embedding_table(idx)
        pos = self.position_embedding_table(torch.arange(T, device=self.device))
        x   = self.blocks(tok + pos)
        x   = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(B * T, -1),
                targets.view(B * T)
            )
        return logits, loss

    def generate(self, idx, max_new_tokens, temperature=0.9, top_k=50, top_p=0.9):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1, None]] = -float("inf")

            probs = F.softmax(logits, dim=-1)

            if top_p < 1.0:
                sorted_probs, sorted_idx = torch.sort(probs, descending=True)
                cumulative = torch.cumsum(sorted_probs, dim=-1)
                mask = cumulative - sorted_probs > top_p
                sorted_probs[mask] = 0
                probs = torch.zeros_like(probs).scatter_(1, sorted_idx, sorted_probs)
                probs = probs / probs.sum(dim=-1, keepdim=True)

            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [15]:
class ModelManager:
    def __init__(self, config, vocab_size):
        self.config = config
        self.vocab_size = vocab_size
        self.model = None

    def build(self):
        self.model = GPT(
            self.vocab_size,
            self.config.n_embd,
            self.config.n_head,
            self.config.n_layer,
            self.config.block_size,
            self.config.dropout,
            self.config.device
        ).to(self.config.device)

        self.model = torch.compile(self.model)

        total_params = sum(p.numel() for p in self.model.parameters())

        print(f"Model parameters : {total_params:,}")
        print(f"Estimated size   : {total_params * 4 / 1e6:.1f} MB (fp32)")

        return self

    def get_model(self):
        return self.model

In [16]:
class Trainer:
    def __init__(self, model, config, batch_loader):
        self.model = model
        self.config = config
        self.batch_loader = batch_loader

        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate
        )

        self.scaler = torch.cuda.amp.GradScaler(enabled=(config.device == "cuda"))

        self.best_val_loss = float("inf")
        self.no_improve_steps = 0

    def get_lr(self, step):
        if step < self.config.warmup_steps:
            return self.config.learning_rate * step / self.config.warmup_steps
        progress = (step - self.config.warmup_steps) / (self.config.max_iters - self.config.warmup_steps)
        return 0.5 * self.config.learning_rate * (1 + math.cos(math.pi * progress))

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        losses = {}

        for split in ["train", "val"]:
            split_losses = torch.zeros(self.config.eval_iters)

            for k in range(self.config.eval_iters):
                x, y = self.batch_loader.get_batch(split)

                with torch.cuda.amp.autocast(enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)

                split_losses[k] = loss.item()

            losses[split] = split_losses.mean().item()

        self.model.train()
        return losses

    def train(self):
        print("\n===== TRAINING START =====")

        for iter in range(self.config.max_iters):
            lr = self.get_lr(iter)
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr

            total_loss = 0.0

            for _ in range(self.config.grad_accum):
                x, y = self.batch_loader.get_batch("train")

                with torch.cuda.amp.autocast(enabled=(self.config.device == "cuda")):
                    _, loss = self.model(x, y)
                    loss = loss / self.config.grad_accum

                self.scaler.scale(loss).backward()
                total_loss += loss.item()

            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad(set_to_none=True)

            if iter % self.config.eval_interval == 0:
                losses = self.evaluate()

                print(f"iter {iter:6d} | "
                      f"train {losses['train']:.4f} | "
                      f"val {losses['val']:.4f} | "
                      f"lr {lr:.6f}")

                if losses["val"] < self.best_val_loss:
                    self.best_val_loss = losses["val"]
                    self.no_improve_steps = 0
                    torch.save(self.model.state_dict(), "best_model.pt")
                    print("✔ best model saved")
                else:
                    self.no_improve_steps += 1

                if self.no_improve_steps >= self.config.patience:
                    print("⛔ Early stopping triggered")
                    break

        print("===== TRAINING COMPLETE =====")

In [17]:
cleaner = TextCleaner("../../Datasets/wikitext-103/wiki.train.tokens")
cleaner.run().report()
text = cleaner.get_text()

[INFO] Loaded dataset : 382,123,643 chars
[INFO] Artifacts fixed
[INFO] Section boundaries marked
[INFO] Lines cleaned
[INFO] Whitespace normalized

[INFO] Final text size : 379,147,541 chars
[INFO] Sample          :
<|endoftext|>
Senjō no Valkyria 3 : Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janu


In [18]:
files = [
    "../../Datasets/wikitext-103/wiki.train.tokens",
    "../../Datasets/wikitext-103/wiki.valid.tokens",
    "../../Datasets/wikitext-103/wiki.test.tokens"
]

output_path = "../../Datasets/wikitext-103/wiki.full.tokens"

In [19]:
total_written = 0

with open(output_path, "w", encoding="utf-8") as outfile:
    for fname in files:
        print("Reading:", fname)

        with open(fname, "r", encoding="utf-8") as infile:
            content = infile.read()

            print(f"  chars read: {len(content):,}")
            outfile.write(content + "\n")

            total_written += len(content)

Reading: ../../Datasets/wikitext-103/wiki.train.tokens
  chars read: 382,123,643
Reading: ../../Datasets/wikitext-103/wiki.valid.tokens
  chars read: 1,140,678
Reading: ../../Datasets/wikitext-103/wiki.test.tokens
  chars read: 1,279,333


In [20]:
cleaner = TextCleaner("../../Datasets/wikitext-103/wiki.full.tokens")
cleaner.run().report()
text = cleaner.get_text()

[INFO] Loaded dataset : 384,543,657 chars
[INFO] Artifacts fixed
[INFO] Section boundaries marked
[INFO] Lines cleaned
[INFO] Whitespace normalized

[INFO] Final text size : 381,535,043 chars
[INFO] Sample          :
<|endoftext|>
Senjō no Valkyria 3 : Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role - playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in Janu


In [21]:
config = Config().summary()


===== CONFIG =====
batch_size     : 16
block_size     : 256
max_iters      : 25000
learning_rate  : 0.0002
n_embd         : 512
n_head         : 8
n_layer        : 8
grad_accum     : 4
dropout        : 0.1
eval_interval  : 500
eval_iters     : 100
warmup_steps   : 1000
patience       : 10
device         : cuda


In [22]:
config.max_iters = 20
config.eval_interval = 5
config.eval_iters = 5

In [23]:
tok_wrapper = TokenizerWrapper("gpt2")
encoded     = tok_wrapper.run(text)
vocab_size  = tok_wrapper.vocab_size

data = torch.tensor(encoded, dtype=torch.long)

[INFO] Tokenizer loaded  : gpt2
[INFO] Vocab size        : 50,257
[INFO] EOS token ID      : 50256
[INFO] Sanity check
       input   : hello world <|endoftext|> next document
       tokens  : [31373, 995, 220, 50256, 1306, 3188]
       decoded : hello world  next document
       EOS ID  : 50256 ✓

[INFO] Encoding text in chunks...
[INFO] Encoding... 100%

===== DATASET ANALYSIS =====
Total Words           : 71,290,798
Total Tokens (BPE)    : 80,137,996
Unique Words          : 254,605
Unique Tokens         : 46,713
Avg Tokens per Word   : 1.12
Avg Word Length       : 4.35


In [24]:
data_manager = DataManager(data).split()
train_data, val_data = data_manager.get_data()

Train tokens : 72,124,196
Val tokens   : 8,013,800
Train size   : 0.58 GB
Val size     : 0.06 GB


In [25]:
config = Config().summary()


===== CONFIG =====
batch_size     : 16
block_size     : 256
max_iters      : 25000
learning_rate  : 0.0002
n_embd         : 512
n_head         : 8
n_layer        : 8
grad_accum     : 4
dropout        : 0.1
eval_interval  : 500
eval_iters     : 100
warmup_steps   : 1000
patience       : 10
device         : cuda


In [26]:
batch_loader = BatchLoader(
    train_data,
    val_data,
    config.block_size,
    config.batch_size,
    config.device
)

In [27]:
model = ModelManager(config, vocab_size).build().get_model()

Model parameters : 51,070,464
Estimated size   : 204.3 MB (fp32)


In [28]:
trainer = Trainer(model, config, batch_loader)
trainer.train()


===== TRAINING START =====


/tmp/ipykernel_3941509/3786605812.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=(config.device == "cuda"))
/tmp/ipykernel_3941509/3786605812.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(self.config.device == "cuda")):
/tmp/ipykernel_3941509/3786605812.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(self.config.device == "cuda")):
/tmp/ipykernel_3941509/3786605812.py:34: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(self.config.device == "cuda")):


iter      0 | train 10.9329 | val 10.9338 | lr 0.000000
✔ best model saved
iter    500 | train 6.6844 | val 6.6899 | lr 0.000100
✔ best model saved
iter   1000 | train 5.8093 | val 5.8173 | lr 0.000200
✔ best model saved
iter   1500 | train 5.2653 | val 5.3170 | lr 0.000200
✔ best model saved
iter   2000 | train 4.9737 | val 4.9867 | lr 0.000199
✔ best model saved
iter   2500 | train 4.7392 | val 4.7751 | lr 0.000198
✔ best model saved
iter   3000 | train 4.5557 | val 4.6338 | lr 0.000197
✔ best model saved
iter   3500 | train 4.3991 | val 4.4567 | lr 0.000195
✔ best model saved
iter   4000 | train 4.2857 | val 4.3514 | lr 0.000192
✔ best model saved
iter   4500 | train 4.2222 | val 4.2348 | lr 0.000190
✔ best model saved
iter   5000 | train 4.1043 | val 4.1822 | lr 0.000187
✔ best model saved
iter   5500 | train 4.0378 | val 4.1334 | lr 0.000183
✔ best model saved
iter   6000 | train 3.9704 | val 4.0847 | lr 0.000179
✔ best model saved
iter   6500 | train 3.9144 | val 4.0411 | lr 0.00

In [29]:
torch.save({
    "model_state_dict": model.state_dict(),
    "config": vars(config),
    "vocab_size": vocab_size
}, "../../Models/wikitext_checkpoints.pt")

In [31]:
config_path = os.path.join("../../Configs", "wikitext_configs.json")
import json
with open(config_path, "w") as f:
    json.dump(vars(config), f, indent=4)

print(f"Config saved to {config_path}")

Config saved to ../../Configs/wikitext_configs.json
